In [1]:
import pandas as pd
import numpy as np
from functools import reduce
region='HK'

In [ ]:
for threshold in [8]:
    for scenario in ['cntl', 'tran_albe']:
        df_temp = pd.read_csv(f'../temperature_trend/data_for_figure/{region}_tsa_{scenario}_urban_rural.csv') 
        df_temp['time'] = pd.to_datetime(df_temp['time'])
        df_temp = df_temp[df_temp['time'].dt.year > 2035]
        df_temp['date'] = df_temp['time'].dt.date
        coldsurge_event_list = []
        for var in ['TSA', 'TSA_U', 'TSA_R']:
            df_temp_tsau = df_temp[['date', var]].copy()
            df_tmin_tsau = df_temp_tsau.groupby('date').min().reset_index()
            df_tmin_tsau['is_cold'] = df_tmin_tsau[var] <= threshold
            # Identify consecutive groups
            df_tmin_tsau['group'] = (df_tmin_tsau['is_cold'] != df_tmin_tsau['is_cold'].shift()).cumsum()

            # For each group, count how many consecutive days satisfy TSA < 8
            coldsurge_groups = df_tmin_tsau.groupby('group').filter(lambda x: x['is_cold'].all() and len(x) >= 2)

            # Optionally, extract the start and end dates of each cold surge event
            coldsurge_events = (df_tmin_tsau.groupby('group').filter(lambda x: x['is_cold'].all() and len(x) >= 2)
                               .groupby('group').agg(start=('date', 'first'), end=('date', 'last'), duration=('date', 'count'))
                               .reset_index(drop=True)
                               )
            coldsurge_events['var'] = var
            coldsurge_event_list.append(coldsurge_events)
        df_coldsurge_event_all = pd.concat(coldsurge_event_list, ignore_index=True)    
        output_filename = f'./data_for_figure/{region}_coldsurge_events_{threshold}_{scenario}_urban_rural.csv'
        df_coldsurge_event_all.to_csv(output_filename, index=False) 
        df_coldsurge_event_all.head()

In [22]:
df_list = []
for scenario in ['cntl', 'tran_albe']:
    df_coldsurge_events_all = pd.read_csv(f'./data_for_figure/{region}_coldsurge_events_{threshold}_{scenario}_urban_rural.csv' )
    df_coldsurge_events_all_urban = df_coldsurge_events_all[df_coldsurge_events_all['var']=='TSA_U'].copy()
    df_coldsurge_events_all_urban['year'] = pd.to_datetime(df_coldsurge_events_all_urban['start']).dt.year
    df_annual_coldsurge_days = (df_coldsurge_events_all_urban.groupby(['year', 'var'])['duration'].sum().reset_index(name=f'coldsurge_days_{scenario}'))
    df_list.append(df_annual_coldsurge_days)
df_merged = reduce(
    lambda left, right: pd.merge(left, right, on='year', how='outer'),
    df_list
)

df_merged = df_merged.sort_values('year').reset_index(drop=True)   
df_merged

,year,var_x,coldsurge_days_cntl,var_y,coldsurge_days_tran_albe
0,2036,TSA_U,34,TSA_U,34
1,2037,TSA_U,25,TSA_U,27
2,2038,TSA_U,38,TSA_U,39
3,2039,TSA_U,19,TSA_U,19


In [2]:
def coldsurge_frequency(df_events):
    frequency = df_events.groupby('year').size().reset_index(name='count')
    return frequency

def coldsurge_seasonal_length(df_events): 
    yearly_first_coldsurge = df_events.groupby('year')['start'].min().reset_index()
    yearly_last_coldsurge = df_events.groupby('year')['end'].max().reset_index()
    yearly_coldsurge = pd.merge(yearly_first_coldsurge, yearly_last_coldsurge, on='year')
    yearly_coldsurge['season_length'] = (yearly_coldsurge['end'] - yearly_coldsurge['start']).dt.days
    return yearly_coldsurge[['year', 'season_length']]

def coldsurge_intensity(df_events, df_temperature):
    yearly_coldsurges = df_events.groupby('year')
    intensity = yearly_coldsurges.apply(
        lambda x: df_temperature.loc[(df_temperature['date'] >= x['start'].min()) & 
                              (df_temperature['date'] <= x['end'].max()), var].mean()).reset_index(name='intensity')
    return intensity

def coldsurge_duration(df_events):
    duration = df_events.groupby('year')['duration'].mean().reset_index()
    return duration

In [3]:
var = 'TSA_U'
metric_list = []
for threshold in [8]:
    metric_results = []
    for scenario in ['cntl', 'tran_albe']:
        df_coldsurge_events = pd.read_csv(f'data_for_figure/{region}_coldsurge_events_{threshold}_{scenario}_urban_rural.csv') #_no_sst
        df_coldsurge_events_urban = df_coldsurge_events[df_coldsurge_events['var']==var].copy()
        df_coldsurge_events_urban['start'] = pd.to_datetime(df_coldsurge_events_urban['start'])
        df_coldsurge_events_urban['end'] = pd.to_datetime(df_coldsurge_events_urban['end'])
        df_coldsurge_events_urban['year'] = df_coldsurge_events_urban['start'].dt.year
        coldsurge_metrics = []
        coldsurge_metrics.append(coldsurge_frequency(df_coldsurge_events_urban).set_index('year'))
        coldsurge_metrics.append(coldsurge_seasonal_length(df_coldsurge_events_urban).set_index('year'))
        df_temp = pd.read_csv(f'../temperature_trend/data_for_figure/{region}_tsa_{scenario}_urban_rural.csv') #_no_sst
        df_temp['time'] = pd.to_datetime(df_temp['time'])
        df_temp['date'] = df_temp['time'].dt.date
        df_temp_tsau = df_temp[['date', var]].copy()
        df_tmax_tsau = df_temp_tsau.groupby('date').max().reset_index()
        df_tmax_tsau['is_hot'] = df_tmax_tsau[var] >= threshold
        df_temp_urban = df_tmax_tsau[df_tmax_tsau['is_hot']].copy()
        df_temp_urban['date'] = pd.to_datetime(df_temp_urban['date'])
        df_temp_urban['year'] = df_temp_urban['date'].dt.year
        coldsurge_metrics.append(coldsurge_intensity(df_coldsurge_events_urban, df_temp_urban).set_index('year'))
        coldsurge_metrics.append(coldsurge_duration(df_coldsurge_events_urban).set_index('year'))
        df_coldsurge_metrics_urban = pd.concat(coldsurge_metrics, axis=1)
        metric_results.append(df_coldsurge_metrics_urban)
    df_coldsurge_metrics_urban_all = pd.concat(metric_results, keys=['cntl', 'tran_albe'], names=['scenario', 'year']).reset_index()  
    df_coldsurge_metrics_urban_all['totoal_days'] = (df_coldsurge_metrics_urban_all['count'] * df_coldsurge_metrics_urban_all['duration'])
    df_coldsurge_metrics_urban_all['exposure'] = (df_coldsurge_metrics_urban_all['count'] * df_coldsurge_metrics_urban_all['duration']) * df_coldsurge_metrics_urban_all['intensity']
    df_coldsurge_metrics_urban_all = np.round(df_coldsurge_metrics_urban_all, 1)
    df_coldsurge_metrics_urban_all['threshold'] = threshold
    metric_list.append(df_coldsurge_metrics_urban_all)
df_metric = pd.concat(metric_list, axis=0, ignore_index=True)
df_metric

/tmp/ipykernel_2670952/879060022.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  intensity = yearly_coldsurges.apply(
/tmp/ipykernel_2670952/879060022.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  intensity = yearly_coldsurges.apply(


,scenario,year,count,season_length,intensity,duration,totoal_days,exposure,threshold
0,cntl,2036,4,355,29.3,8.5,34.0,997.5,8
1,cntl,2037,8,346,29.9,3.1,25.0,748.7,8
2,cntl,2038,9,350,29.6,4.2,38.0,1125.7,8
3,cntl,2039,5,350,29.8,3.8,19.0,566.8,8
4,tran_albe,2036,4,355,29.3,8.5,34.0,994.8,8
5,tran_albe,2037,9,346,29.8,3.0,27.0,804.7,8
6,tran_albe,2038,9,351,29.4,4.3,39.0,1146.5,8
7,tran_albe,2039,5,350,29.6,3.8,19.0,561.8,8
